In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import sparrow as sp
import spatialdata as sd

the value of the environment variable BASIC_DCT_BACKEND is not in ["JAX","SCIPY"]
2024-10-25 11:42:44,381 - sparrow.image.pixel_clustering._clustering - WARNING - 'flowsom' not installed, to use 'sp.im.flowsom', please install this library.
2024-10-25 11:42:46,333 - sparrow.table.cell_clustering._clustering - WARNING - 'flowsom' not installed, to use 'sp.tb.flowsom', please install this library.
2024-10-25 11:42:46,440 - sparrow.plot._qc_cells - WARNING - 'joypy' not installed, to use 'sp.pl.ridgeplot_channel' and 'sp.pl.ridgeplot_channel_sample', please install this library.
2024-10-25 11:42:46,443 - sparrow.plot._qc_image - WARNING - 'textalloc' not installed, to use 'sp.pl.group_snr_ratio' and 'sp.pl.snr_ratio', please install this library.


## OPTIONAL: Read sdata

In [3]:
import dask.array as da
import os
from spatialdata import SpatialData
from tifffile import imread

output_path = r'D:\Data'

img = imread( r'cropped_image.tiff' )

sdata=SpatialData()
sdata.write( os.path.join( output_path, "sdata_segmentation_test.zarr" ) )

sdata=sp.im.add_image_layer(
    sdata,
    arr = da.asarray(img).transpose( 2,0,1 ),
    output_layer = "raw_image",
    overwrite=True,
        )

INFO     The Zarr backing store has been changed from None the new file path: D:\Data\sdata_segmentation_test.zarr 


2024-10-25 11:42:46,786 - sparrow.image._manager - WARNING - No dims parameter specified. Assuming order of dimension of provided array is ((c), (z), y, x)
2024-10-25 11:42:46,796 - sparrow.image._manager - INFO - Writing results to layer 'raw_image'
c:\Users\julienm\miniconda3\envs\harpy\lib\site-packages\zarr\creation.py:300: UserWarning: ignoring keyword argument 'read_only'
  warn(f"ignoring keyword argument {k!r}")


In [9]:
import os
from pathlib import Path

import torch
from instanseg.utils.utils import _choose_device, download_model

# Call the function to download and extract the models

model_to_download = "brightfield_nuclei"  # "fluorescence_nuclei_and_cells" or "brightfield_nuclei"
download_model(model_to_download)

path_to_torchscript_model = f'c:/Users/julienm/miniconda3/envs/harpy/lib/site-packages/instanseg/bioimageio_models/{model_to_download}/instanseg.pt'

instanseg_model = torch.jit.load(path_to_torchscript_model)

device = _choose_device()
instanseg_model.to(device)
print("Model loaded")

Model brightfield_nuclei downloaded and extracted to c:\Users\julienm\miniconda3\envs\harpy\lib\site-packages\instanseg\utils\../bioimageio_models/
Requesting default device: cpu
Model loaded


In [20]:
from sparrow.image.segmentation.segmentation_models._instanseg import _instanseg

sdata=sp.im.segment(
    sdata=sdata,
    img_layer="raw_image",
    # Only specify one labels layer if output parameter is set to "whole_cell" or "nuclei".
    output_labels_layer="segmentation_mask_nuclei",
    output_shapes_layer=None,
    # if parameter labels_layer_align is specified, labels in "segmentation_mask_nuclei" will be aligned with corresponding labels of whole cell masks in "segmentation_mask_whole_cell"
    # labels_layer_align="segmentation_mask_whole_cell",
    model=_instanseg,
    # Set output to "nucleus" to get only the nucleus mask, "whole_cell" to get the whole cell mask and 'all' or None to get both "nuclei" and "whole_cell"
    output="all",
    chunks=(512, 512),
    device=device,
    instanseg_model=instanseg_model,
    overwrite=True,
)

c:\Users\julienm\miniconda3\envs\harpy\lib\site-packages\zarr\creation.py:300: UserWarning: ignoring keyword argument 'read_only'
  warn(f"ignoring keyword argument {k!r}")
2024-10-25 13:46:13,594 - sparrow.image._manager - WARNING - No dims parameter specified. Assuming order of dimension of provided array is ((c), (z), y, x)
2024-10-25 13:46:13,610 - sparrow.image._manager - INFO - Writing results to layer 'segmentation_mask_nuclei'
c:\Users\julienm\miniconda3\envs\harpy\lib\site-packages\zarr\creation.py:300: UserWarning: ignoring keyword argument 'read_only'
  warn(f"ignoring keyword argument {k!r}")


In [ ]:
sdata

In [21]:
from napari_spatialdata import Interactive
Interactive(sdata)

c:\Users\julienm\miniconda3\envs\harpy\lib\site-packages\anndata\_core\aligned_df.py:68: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)
2024-10-25 13:46:33.564 | INFO     | napari_spatialdata._view:_on_layer_update:355 - Updating layer.
2024-10-25 13:46:33.565 | INFO     | napari_spatialdata._view:_on_layer_update:355 - Updating layer.
2024-10-25 13:46:35.459 | INFO     | napari_spatialdata._view:_on_layer_update:355 - Updating layer.
2024-10-25 13:46:35.463 | INFO     | napari_spatialdata._view:_on_layer_update:355 - Updating layer.


In [22]:
0.22 / 0.6

0.3666666666666667

In [23]:
from skimage import io
from skimage.transform import rescale
import tifffile

# Load the image
image = io.imread('cropped_image.tiff')

# Original resolution: 0.22 micrometer per pixel
# Target resolution: 0.5 micrometer per pixel
# Calculate the downscale factor
downscale_factor = 0.22 / 0.5

# Downsample the image
downsampled_image = rescale(image, downscale_factor, anti_aliasing=True, channel_axis=-1)

# Save the downsampled image as a TIFF file with the scale factor set to 0.5 micrometer per pixel
tifffile.imwrite('downsampled_image_HE.tiff', (downsampled_image * 255).astype('uint8'), resolution=(1/0.5, 1/0.5),resolutionunit=None)

print("The image has been downsampled and saved as 'downsampled_image_HE.tiff' with a scale factor of 0.5 micrometer per pixel.")


The image has been downsampled and saved as 'downsampled_image_HE_2.tiff' with a scale factor of 0.5 micrometer per pixel.
